# 🔧 Cookbook 2: Creating Custom Tools - From Basics to Advanced

In this cookbook, you'll master creating custom tools for Strands Agents!

**Topics covered:**
1. The `@tool` decorator basics
2. Type hints and docstrings
3. **ToolContext** - Accessing agent state and session info
4. Class-based tools for stateful operations
5. Async tools and streaming
6. Advanced patterns (custom return types, error handling)

**Prerequisites**: Complete Cookbook 1 first!

---

In [ ]:
# Install packages if needed
!pip install strands-agents strands-agents-tools -q
print("✅ Ready to go!")

---
## Part 1: Tool Basics
---

## 🎯 Why Custom Tools?

Built-in tools are great, but sometimes you need:
- Integration with your own APIs
- Custom business logic
- Domain-specific functionality
- Access to request context (user IDs, session data)

With the `@tool` decorator, you can turn any Python function into a tool!

### 📝 Your First Custom Tool

Let's create a simple word counter tool.

In [ ]:
from strands import Agent, tool

@tool
def word_count(text: str) -> int:
    """Count the number of words in the given text.
    
    Args:
        text: The text to count words in.
    
    Returns:
        The number of words in the text.
    """
    return len(text.split())

# Create an agent with our custom tool
agent = Agent(tools=[word_count])

# Test it!
response = agent("How many words are in: 'The quick brown fox jumps over the lazy dog'?")

print("\n✅ Our custom tool worked!")

### 🔍 Understanding the Tool Structure

```python
@tool
def my_tool(param1: str, param2: int = 10) -> str:
    """Short description of what the tool does.
    
    This longer description helps the LLM understand
    when and how to use this tool.
    
    Args:
        param1: Description of param1
        param2: Description of param2 (optional)
    
    Returns:
        Description of the return value
    """
    return result
```

**Key components:**
1. **Type hints** - Tell the LLM what types to use
2. **Docstring** - Helps the LLM understand the tool
3. **Args section** - Describes each parameter
4. **Return type** - What the tool returns

---
## Part 2: ToolContext - Accessing Agent and Session State ⭐
---

One of the most powerful features of Strands tools is **ToolContext**. It lets your tools access:
- The invoking agent instance
- Current tool use information
- Invocation state (user IDs, session data, custom objects)

This is **essential** for building production tools!

### 🔐 Basic ToolContext Usage

To use ToolContext, set `context=True` in the decorator and add a `tool_context` parameter:

In [ ]:
from strands import Agent, tool, ToolContext

@tool(context=True)
def get_agent_info(tool_context: ToolContext) -> str:
    """Get information about the current agent.
    
    Args:
        tool_context: The context object (automatically injected).
    
    Returns:
        Information about the agent.
    """
    agent_name = tool_context.agent.name or "Unnamed Agent"
    message_count = len(tool_context.agent.messages)
    
    return f"Agent '{agent_name}' has {message_count} messages in history"

@tool(context=True)
def get_tool_use_id(tool_context: ToolContext) -> str:
    """Get the current tool use ID.
    
    Args:
        tool_context: The context object.
    
    Returns:
        The tool use ID.
    """
    return f"Tool use ID: {tool_context.tool_use['toolUseId']}"

# Create an agent with a name
agent = Agent(
    tools=[get_agent_info, get_tool_use_id],
    name="ContextDemo"
)

# Test it
response = agent("What's your name and how many messages do you have?")
print("\n✅ ToolContext working!")

### 🎯 Accessing Invocation State - The Key Pattern!

**Invocation state** is how you pass request-specific data to tools without including it in prompts. This is crucial for:
- User IDs and session IDs
- Database connections
- API keys and auth tokens
- Custom configuration

In [ ]:
from strands import Agent, tool, ToolContext

@tool(context=True)
def get_user_profile(tool_context: ToolContext) -> dict:
    """Get the current user's profile information.
    
    Args:
        tool_context: The context object containing user information.
    
    Returns:
        The user's profile data.
    """
    # Access invocation state - this is passed when calling the agent!
    user_id = tool_context.invocation_state.get("user_id", "anonymous")
    user_name = tool_context.invocation_state.get("user_name", "Unknown")
    user_tier = tool_context.invocation_state.get("user_tier", "free")
    
    return {
        "user_id": user_id,
        "name": user_name,
        "tier": user_tier,
        "message": f"Hello {user_name}! You're on the {user_tier} plan."
    }

@tool(context=True)
def make_api_call(endpoint: str, tool_context: ToolContext) -> dict:
    """Make an API call with user context.
    
    Args:
        endpoint: The API endpoint to call.
        tool_context: Context containing auth info.
    
    Returns:
        Simulated API response.
    """
    # Get auth from invocation state
    api_key = tool_context.invocation_state.get("api_key", "no-key")
    user_id = tool_context.invocation_state.get("user_id", "anonymous")
    
    # Simulate API call with auth
    return {
        "endpoint": endpoint,
        "authenticated": api_key != "no-key",
        "user_id": user_id,
        "response": f"Data from {endpoint} for user {user_id}"
    }

# Create agent
agent = Agent(tools=[get_user_profile, make_api_call])

# Call the agent WITH invocation state - this is the magic!
response = agent(
    "What's my profile information and then call the /users endpoint?",
    user_id="user_12345",
    user_name="Alice",
    user_tier="premium",
    api_key="secret-key-abc123"
)

print("\n✅ Invocation state accessed successfully!")

### 💡 Invocation State vs Tool Parameters vs Class State

When should you use what?

| Approach | Use When | Example |
|----------|----------|--------|
| **Tool Parameters** | LLM should reason and provide the value | Search queries, file paths |
| **Invocation State** | Context that varies per request | User IDs, session tokens |
| **Class State** | Configuration that doesn't change between requests | API endpoints, DB connections |

### 🎨 Custom Context Parameter Name

You can customize the parameter name if you prefer something shorter:

In [ ]:
from strands import Agent, tool, ToolContext

# Use 'ctx' instead of 'tool_context'
@tool(context="ctx")
def quick_check(ctx: ToolContext) -> str:
    """Quickly check the current context.
    
    Args:
        ctx: Shortened context parameter name.
    
    Returns:
        Context information.
    """
    user = ctx.invocation_state.get("user", "guest")
    return f"Hello {user}! Tool ID: {ctx.tool_use['toolUseId'][:8]}..."

agent = Agent(tools=[quick_check])
response = agent("Do a quick check", user="Bob")
print("\n✅ Custom context parameter name works!")

---
## Part 3: Class-Based Tools for Stateful Operations
---

When you need to share resources between multiple tools (like database connections), use class-based tools.

In [ ]:
from strands import Agent, tool

class DatabaseTools:
    """A set of database-related tools that share a connection."""
    
    def __init__(self, connection_string: str):
        """Initialize with a connection string."""
        self.connection_string = connection_string
        self.connected = True
        # In real code: self.connection = database.connect(connection_string)
        print(f"📊 Connected to: {connection_string}")
    
    @tool
    def query_database(self, sql: str) -> dict:
        """Run a SQL query against the database.
        
        Args:
            sql: The SQL query to execute.
        
        Returns:
            Query results.
        """
        # Uses the shared connection from self
        return {
            "query": sql,
            "connection": self.connection_string,
            "results": f"[Simulated results for: {sql}]"
        }
    
    @tool
    def insert_record(self, table: str, data: dict) -> str:
        """Insert a record into the database.
        
        Args:
            table: The table name.
            data: The data to insert.
        
        Returns:
            Confirmation message.
        """
        return f"✅ Inserted into {table}: {data}"
    
    @tool
    def get_connection_status(self) -> str:
        """Get the current database connection status.
        
        Returns:
            Connection status message.
        """
        status = "connected" if self.connected else "disconnected"
        return f"Database: {self.connection_string} - Status: {status}"

# Create tools instance with configuration
db_tools = DatabaseTools("postgresql://localhost:5432/myapp")

# Pass the bound methods to the agent
agent = Agent(tools=[
    db_tools.query_database,
    db_tools.insert_record,
    db_tools.get_connection_status
])

# Test it
response = agent("First check the connection status, then query for all users")
print("\n✅ Class-based tools working!")

---
## Part 4: Async Tools and Streaming
---

For I/O-bound operations (API calls, file operations), use async tools!

In [ ]:
import asyncio
from strands import Agent, tool

@tool
async def fetch_weather(city: str) -> dict:
    """Fetch weather data for a city (simulated async call).
    
    Args:
        city: The city to get weather for.
    
    Returns:
        Weather data.
    """
    # Simulate API call delay
    await asyncio.sleep(0.5)
    
    return {
        "city": city,
        "temperature": 72,
        "conditions": "Sunny",
        "humidity": 45
    }

@tool
async def fetch_news(topic: str) -> dict:
    """Fetch news headlines (simulated async call).
    
    Args:
        topic: The topic to search for.
    
    Returns:
        News headlines.
    """
    await asyncio.sleep(0.5)
    
    return {
        "topic": topic,
        "headlines": [
            f"Breaking: New developments in {topic}",
            f"{topic} trends continue to grow"
        ]
    }

# Async tools are invoked concurrently when possible!
agent = Agent(tools=[fetch_weather, fetch_news])

# This will call both tools concurrently
async def main():
    result = await agent.invoke_async(
        "What's the weather in Seattle and what's the news about AI?"
    )
    return result

# Run it
result = asyncio.run(main())
print("\n✅ Async tools executed (concurrently when possible)!")

### 📡 Streaming Progress from Tools

Tools can yield intermediate results for real-time progress updates:

In [ ]:
import asyncio
from datetime import datetime
from strands import Agent, tool

@tool
async def process_dataset(records: int) -> str:
    """Process a dataset with progress updates.
    
    Args:
        records: Number of records to process.
    
    Returns:
        Final completion message.
    """
    start = datetime.now()
    
    for i in range(records):
        await asyncio.sleep(0.1)  # Simulate work
        
        # Yield progress updates every 5 records
        if i % 5 == 0 and i > 0:
            elapsed = (datetime.now() - start).total_seconds()
            yield f"Processed {i}/{records} records ({elapsed:.1f}s elapsed)"
    
    # Final result
    total_time = (datetime.now() - start).total_seconds()
    return f"✅ Completed processing {records} records in {total_time:.1f}s"

agent = Agent(tools=[process_dataset])

# Stream the results
async def stream_processing():
    print("Starting processing with streaming...\n")
    async for event in agent.stream_async("Process 20 records"):
        if tool_stream := event.get("tool_stream_event"):
            if data := tool_stream.get("data"):
                print(f"📊 Progress: {data}")

asyncio.run(stream_processing())
print("\n✅ Streaming complete!")

---
## Part 5: Advanced Tool Features
---

### 🎛️ Custom Return Types

For more control over responses, return a ToolResult dictionary:

In [ ]:
from strands import Agent, tool

@tool
def fetch_data(source_id: str) -> dict:
    """Fetch data from a specified source.
    
    Args:
        source_id: Identifier for the data source.
    
    Returns:
        Structured response with status and content.
    """
    try:
        # Simulate fetching data
        if source_id == "error":
            raise ValueError("Invalid source ID")
        
        data = {"id": source_id, "value": 42, "timestamp": "2024-01-15"}
        
        # Return structured ToolResult
        return {
            "status": "success",
            "content": [
                {"text": f"Successfully fetched from {source_id}"},
                {"json": data}  # Include structured data!
            ]
        }
    except Exception as e:
        # Return error response
        return {
            "status": "error",
            "content": [
                {"text": f"Error fetching data: {str(e)}"}
            ]
        }

agent = Agent(tools=[fetch_data])
response = agent("Fetch data from source ABC123")
print("\n✅ Custom return type working!")

### 🏷️ Overriding Tool Name and Description

You can customize how your tool appears to the LLM:

In [ ]:
from strands import Agent, tool

@tool(
    name="weather_lookup",
    description="Look up current weather conditions for any city worldwide"
)
def get_weather_internal(city: str) -> str:
    """Internal implementation function.
    
    Args:
        city: City name.
    """
    return f"Weather in {city}: Sunny, 75°F"

agent = Agent(tools=[get_weather_internal])

# The agent sees "weather_lookup" not "get_weather_internal"
print(f"Tool names: {agent.tool_names}")
response = agent("What's the weather in Paris?")

---
## 🏆 Challenge: Build a Complete Tool Set

Create a `UserManagement` class with tools that:
1. Use `ToolContext` to access user authentication
2. Support multiple operations (get user, update user, delete user)
3. Return proper ToolResult structures with error handling

In [ ]:
from strands import Agent, tool, ToolContext

class UserManagement:
    """Complete user management tool set."""
    
    def __init__(self):
        # Simulated database
        self.users = {
            "user_1": {"name": "Alice", "email": "alice@example.com", "role": "admin"},
            "user_2": {"name": "Bob", "email": "bob@example.com", "role": "user"},
        }
    
    @tool(context=True)
    def get_user(self, user_id: str, tool_context: ToolContext) -> dict:
        """Get a user's information.
        
        Args:
            user_id: The ID of the user to retrieve.
            tool_context: Context with auth info.
        
        Returns:
            User data or error.
        """
        # Check authorization from invocation state
        requester_role = tool_context.invocation_state.get("role", "guest")
        
        if user_id in self.users:
            user = self.users[user_id].copy()
            # Only show email to admins
            if requester_role != "admin":
                user.pop("email", None)
            return {"status": "success", "content": [{"json": user}]}
        else:
            return {"status": "error", "content": [{"text": f"User {user_id} not found"}]}
    
    @tool(context=True)
    def update_user(self, user_id: str, updates: dict, tool_context: ToolContext) -> dict:
        """Update a user's information.
        
        Args:
            user_id: The ID of the user to update.
            updates: Dictionary of fields to update.
            tool_context: Context with auth info.
        
        Returns:
            Success or error message.
        """
        requester_role = tool_context.invocation_state.get("role", "guest")
        
        if requester_role != "admin":
            return {"status": "error", "content": [{"text": "Only admins can update users"}]}
        
        if user_id not in self.users:
            return {"status": "error", "content": [{"text": f"User {user_id} not found"}]}
        
        self.users[user_id].update(updates)
        return {"status": "success", "content": [{"text": f"Updated user {user_id}"}]}

# Test it
user_mgmt = UserManagement()
agent = Agent(tools=[user_mgmt.get_user, user_mgmt.update_user])

# As a regular user
print("As a regular user:")
response = agent("Get info for user_1", role="user")

print("\nAs an admin:")
response = agent("Get info for user_1 and update their name to 'Alicia'", role="admin")

---
## 📚 Summary

In this cookbook, you learned:

1. **Basic tools** with `@tool` decorator, type hints, and docstrings
2. **ToolContext** for accessing agent state and invocation data (⭐ key feature!)
3. **Invocation state** for passing user IDs, auth tokens, and custom objects
4. **Class-based tools** for sharing state between related tools
5. **Async tools** for concurrent I/O operations
6. **Streaming** for real-time progress updates
7. **Custom return types** with ToolResult structures

**Next**: Check out Cookbook 3 for conversations, memory, and multi-agent systems!